# Entropic, Burg, and Quadratic Regularized Couplings

This notebook illustrates how the choice of entropy changes the geometry of a balanced regularized OT plan.  For a reference density $a_i b_j$ and the coupling density
$$
r_{ij}=\frac{P_{ij}}{a_i b_j},
$$
the first-order condition takes the common form
$$
h'(r_{ij})=\frac{f_i+g_j-C_{ij}}{\lambda}.
$$
We compare three scalar entropies: KL, Burg, and a quadratic density penalty.  The goal is a visual explanation of the plans, not a solver benchmark.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    LIGHT_GRAY,
    AXIS_LINE_WIDTH,
    box_axes,
    figure_dir,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "sinkhorn-entropic-versus-quadratic-regularization"
OUT = figure_dir(NAME)


## A common one-dimensional mixture problem

Both marginals are smooth Gaussian mixtures sampled on the same number of grid points.  The quadratic cost is normalized by its median so that a single regularization scale gives readable, deliberately smoothed couplings for all three entropies.


In [ ]:
def gaussian(z, mean, sigma):
    return np.exp(-0.5 * ((z - mean) / sigma) ** 2)


def normalize(w):
    w = np.maximum(np.asarray(w, dtype=float), 0.0)
    return w / w.sum()


n = 58
x = np.linspace(-3.0, 3.05, n)
y = np.linspace(-2.85, 3.20, n)
a = normalize(0.55 * gaussian(x, -1.38, 0.36) + 0.45 * gaussian(x, 0.58, 0.48))
b = normalize(0.34 * gaussian(y, -0.62, 0.34) + 0.66 * gaussian(y, 1.38, 0.52))
C = (x[:, None] - y[None, :]) ** 2
C = C / np.median(C)
LAMBDA = 0.45


## Three entropy choices

KL gives the usual multiplicative Sinkhorn scaling.  Burg acts as a barrier and keeps a full positive support with a different tail profile.  The quadratic penalty has a positive-part formula, so it can keep exact zeros even at the same regularization scale.


In [ ]:
def kl_plan(C, a, b, lam, n_iter=900):
    K = np.exp(-C / lam)
    u = np.ones_like(a)
    v = np.ones_like(b)
    for _ in range(n_iter):
        u = 1.0 / np.maximum(K @ (b * v), 1e-300)
        v = 1.0 / np.maximum(K.T @ (a * u), 1e-300)
    return (a[:, None] * b[None, :]) * (u[:, None] * K * v[None, :])


def solve_bisection(value, lo, hi, target=1.0, n_steps=48):
    for _ in range(n_steps):
        mid = 0.5 * (lo + hi)
        if value(mid) > target:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)


def regularized_density_plan(C, a, b, lam, kind, n_iter=140):
    f = np.zeros_like(a)
    g = np.zeros_like(b)
    span = float(np.ptp(C) + 8.0 * lam + 1.0)

    def density(S):
        if kind == "quadratic":
            return np.maximum(0.0, 1.0 + S / lam)
        if kind == "burg":
            return 1.0 / np.maximum(1.0 - S / lam, 1e-12)
        raise ValueError(kind)

    for _ in range(n_iter):
        for i in range(len(a)):
            if kind == "burg":
                hi = float(np.min(C[i] - g + lam) - 1e-11)
                lo = hi - span
                while np.sum(b * density(lo + g - C[i])) > 1.0:
                    lo -= span
            else:
                lo = float(np.min(C[i] - g) - span)
                hi = float(np.max(C[i] - g) + span)
                while np.sum(b * density(lo + g - C[i])) > 1.0:
                    lo -= span
                while np.sum(b * density(hi + g - C[i])) < 1.0:
                    hi += span
            f[i] = solve_bisection(lambda t, i=i: np.sum(b * density(t + g - C[i])), lo, hi)
        shift = np.average(f, weights=a)
        f -= shift
        g += shift

        for j in range(len(b)):
            if kind == "burg":
                hi = float(np.min(C[:, j] - f + lam) - 1e-11)
                lo = hi - span
                while np.sum(a * density(f + lo - C[:, j])) > 1.0:
                    lo -= span
            else:
                lo = float(np.min(C[:, j] - f) - span)
                hi = float(np.max(C[:, j] - f) + span)
                while np.sum(a * density(f + lo - C[:, j])) > 1.0:
                    lo -= span
                while np.sum(a * density(f + hi - C[:, j])) < 1.0:
                    hi += span
            g[j] = solve_bisection(lambda t, j=j: np.sum(a * density(f + t - C[:, j])), lo, hi)
        shift = np.average(g, weights=b)
        g -= shift
        f += shift

    R = density(f[:, None] + g[None, :] - C)
    return (a[:, None] * b[None, :]) * R


P_kl = kl_plan(C, a, b, LAMBDA)
P_burg = regularized_density_plan(C, a, b, LAMBDA, "burg", n_iter=430)
P_quad = regularized_density_plan(C, a, b, LAMBDA, "quadratic", n_iter=160)
plans = {"kl": P_kl, "burg": P_burg, "quadratic": P_quad}

for name, P in plans.items():
    row_err = np.linalg.norm(P.sum(axis=1) - a, 1)
    col_err = np.linalg.norm(P.sum(axis=0) - b, 1)
    print(f"{name:9s} row error {row_err:.2e}  col error {col_err:.2e}  support {(P > 1e-8).mean():.2f}")


## Matrix rendering with side marginals

Each PDF contains only one regularized plan and the two fixed marginals.  Titles are intentionally omitted from the PDFs; the LaTeX document supplies the panel names and caption.


In [ ]:
cmap = LinearSegmentedColormap.from_list(
    "ot4ml_plan_violet",
    ["#ffffff", "#eee8dc", "#c6a0d0", VIOLET],
)
V_MAX = max(np.sqrt(P).max() for P in plans.values())


def draw_marginal_matrix(P, path):
    fig = plt.figure(figsize=(2.18, 2.18))
    gs = fig.add_gridspec(
        2,
        2,
        width_ratios=(0.24, 1.0),
        height_ratios=(0.24, 1.0),
        left=0.04,
        right=0.99,
        bottom=0.04,
        top=0.99,
        wspace=0.015,
        hspace=0.015,
    )
    ax_empty = fig.add_subplot(gs[0, 0])
    ax_top = fig.add_subplot(gs[0, 1])
    ax_left = fig.add_subplot(gs[1, 0])
    ax = fig.add_subplot(gs[1, 1])
    ax_empty.axis("off")

    ax.imshow(
        np.sqrt(P),
        origin="lower",
        extent=(y.min(), y.max(), x.min(), x.max()),
        cmap=cmap,
        vmin=0,
        vmax=V_MAX,
        interpolation="bilinear",
        aspect="auto",
    )
    ax.set_xlim(y.min(), y.max())
    ax.set_ylim(x.min(), x.max())
    ax.set_xticks([])
    ax.set_yticks([])
    box_axes(ax)

    beta_curve = b / b.max()
    ax_top.fill_between(y, 0, beta_curve, color=BLUE, alpha=0.18, linewidth=0)
    ax_top.plot(y, beta_curve, color=BLUE, lw=0.95)
    ax_top.set_xlim(y.min(), y.max())
    ax_top.set_ylim(0, 1.08)
    ax_top.axis("off")

    alpha_curve = a / a.max()
    ax_left.fill_betweenx(x, 0, alpha_curve, color=RED, alpha=0.18, linewidth=0)
    ax_left.plot(alpha_curve, x, color=RED, lw=0.95)
    ax_left.set_ylim(x.min(), x.max())
    ax_left.set_xlim(1.08, 0)
    ax_left.axis("off")

    save_pdf(fig, path, pad_inches=0.025)
    plt.close(fig)


draw_marginal_matrix(P_kl, OUT / "kl-plan.pdf")
draw_marginal_matrix(P_burg, OUT / "burg-plan.pdf")
draw_marginal_matrix(P_quad, OUT / "quadratic-plan.pdf")
